# Construction of Combined Label Splits with a 64-Day History Guard

## Objective
This notebook constructs a single combined wildfire label table from the finalized fire and no-fire pattern pools. The output is designed for downstream modeling workflows that require a split assignment at the sample-key level.

## Split Design
The split policy is year-based and summer-focused. Validation samples are drawn from designated summer years, the test split is assigned to the latest summer block, and the training split contains all remaining eligible rows.

## History Guard
A strict 64-day lookback guard is applied after split assignment. A sample is retained only if its full 64-day history remains inside the same split period. This prevents train/validation/test leakage through historical context.

## Output
The notebook writes a single combined label file:
- `LABELS_combined.csv`

The exported table contains `target_date`, `row`, `col`, `label`, `type_class`, and `split`.


## Environment and Imports

This section loads the standard analysis libraries used for file handling, random sampling, and summary export.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd


## Configuration

This section defines the raw class files, the split policy, the 64-day history constraint, and the optional class-ratio balancing parameters used after split filtering.


In [2]:
# -------------------------
# Config
# -------------------------
BASE = Path('/Users/copperhead07/Computing/DataCollection/RealWork')
LABEL_DIR = BASE / 'LABEL' / 'pattern_matches'

RAW_FILES = {
    'FIRETYPE1': 'pattern_fire_1.csv',
    'FIRETYPE5': 'pattern_fire_5.csv',
    'FIRETYPE2': 'pattern_fre_2.csv',
    'NOFIRE0A': 'nofire_0A.csv',
    'NOFIRE0B': 'nofire_0B.csv',
    'NOFIRE0C': 'nofire_0C.csv',
    'NOFIRE0D': 'nofire_0D.csv',
}

SEED = 42
HISTORY_DAYS = 64
SUMMER_MONTHS = (5, 6, 7, 8)

# New split policy
# - test: summer 2023
# - val: summer of selected years (auto-expanded to target rows)
# - train: everything else
VAL_SUMMER_YEARS = (2021, 2022)
TEST_SUMMER_YEAR = 2023
AUTO_SELECT_VAL_YEARS = True
TARGET_VAL_ROWS = 10_000

# Optional ratio rebalance after split+history filtering
APPLY_RATIO = True
FIRE_RATIO_232 = {'FIRETYPE1': 2, 'FIRETYPE5': 3, 'FIRETYPE2': 2}
NOFIRE_RATIO_1121 = {'NOFIRE0A': 1, 'NOFIRE0B': 1, 'NOFIRE0C': 2, 'NOFIRE0D': 1}
ENFORCE_FIRE_NOFIRE_1_TO_1 = True

# Prefer summer rows while downsampling train for ratio matching.
# 1.0 = no preference, >1.0 gives higher chance to summer rows.
TRAIN_SUMMER_WEIGHT = 2.0

OUT_LABELS = LABEL_DIR / 'LABELS_combined.csv'
OUT_SUMMARY = LABEL_DIR / 'label_combined_summary.json'

KEY_COLS = ['target_date', 'row', 'col']
OUT_COLS = ['target_date', 'row', 'col', 'label', 'type_class', 'split']

for cls, fn in RAW_FILES.items():
    p = LABEL_DIR / fn
    if not p.exists():
        raise FileNotFoundError(p)

print('LABEL_DIR:', LABEL_DIR)
print('OUTPUT:', OUT_LABELS)
print('APPLY_RATIO:', APPLY_RATIO)
print('VAL summer years (seed):', VAL_SUMMER_YEARS)
print('TEST summer year:', TEST_SUMMER_YEAR)
print('AUTO_SELECT_VAL_YEARS:', AUTO_SELECT_VAL_YEARS, '| TARGET_VAL_ROWS:', TARGET_VAL_ROWS)



LABEL_DIR: /Users/copperhead07/Computing/DataCollection/RealWork/LABEL/pattern_matches
OUTPUT: /Users/copperhead07/Computing/DataCollection/RealWork/LABEL/pattern_matches/LABELS_combined.csv
APPLY_RATIO: True
VAL summer years (seed): (2021, 2022)
TEST summer year: 2023
AUTO_SELECT_VAL_YEARS: True | TARGET_VAL_ROWS: 10000


## Helper Functions

The helper functions in this section standardize raw class tables, implement weighted sampling, and perform optional ratio-based rebalancing within each split.


In [3]:
# -------------------------
# Helpers
# -------------------------
def _load_one(path: Path, type_class: str) -> pd.DataFrame:
    req = ['target_date', 'row', 'col']
    cols = pd.read_csv(path, nrows=0).columns.tolist()
    miss = sorted(set(req) - set(cols))
    if miss:
        raise ValueError(f'{path.name} missing required columns: {miss}')
    df = pd.read_csv(path, usecols=req)
    dt = pd.to_datetime(df['target_date'], utc=True, errors='coerce').dt.floor('D')
    df['target_date'] = dt.dt.strftime('%Y-%m-%d')
    df['row'] = pd.to_numeric(df['row'], errors='coerce')
    df['col'] = pd.to_numeric(df['col'], errors='coerce')
    df = df.dropna(subset=['target_date', 'row', 'col']).copy()
    df['row'] = df['row'].astype(np.int32)
    df['col'] = df['col'].astype(np.int32)
    df['type_class'] = type_class
    df['label'] = 1 if type_class.startswith('FIRE') else 0
    return df[['target_date', 'row', 'col', 'label', 'type_class']]


def _required_per_unit(fire_ratio: dict, nofire_ratio: dict, enforce_1to1: bool = True) -> dict:
    if enforce_1to1:
        s_fire = int(sum(fire_ratio.values()))
        s_nofire = int(sum(nofire_ratio.values()))
        req = {}
        for k, v in fire_ratio.items():
            req[k] = int(v) * s_nofire
        for k, v in nofire_ratio.items():
            req[k] = int(v) * s_fire
        return req
    req = {**{k: int(v) for k, v in fire_ratio.items()}, **{k: int(v) for k, v in nofire_ratio.items()}}
    return req


def _sample_indices_weighted(idxs: np.ndarray, n_take: int, rng: np.random.Generator, weights: np.ndarray | None = None) -> np.ndarray:
    if n_take <= 0:
        return np.array([], dtype=np.int64)
    if len(idxs) < n_take:
        raise ValueError(f'Cannot sample n_take={n_take} from len={len(idxs)} without replacement')
    if weights is None:
        return rng.choice(idxs, size=n_take, replace=False)
    w = np.asarray(weights, dtype=np.float64)
    w = np.clip(w, 0.0, None)
    if np.all(w <= 0):
        return rng.choice(idxs, size=n_take, replace=False)
    p = w / w.sum()
    return rng.choice(idxs, size=n_take, replace=False, p=p)


def _rebalance_per_split(
    labels: pd.DataFrame,
    seed: int,
    fire_ratio: dict,
    nofire_ratio: dict,
    enforce_1to1: bool = True,
    summer_months: tuple[int, ...] = (5, 6, 7, 8),
    train_summer_weight: float = 1.0,
):
    req = _required_per_unit(fire_ratio, nofire_ratio, enforce_1to1)
    class_order = list(req.keys())
    rng = np.random.default_rng(int(seed))
    out = []
    summary = {}

    for split_name in ['train', 'val', 'test']:
        s = labels[labels['split'] == split_name].copy()
        if s.empty:
            summary[split_name] = {'mode': 'empty', 'rows_before': 0, 'rows_after': 0}
            continue

        s['target_dt'] = pd.to_datetime(s['target_date'], utc=True, errors='coerce')
        s['is_summer'] = s['target_dt'].dt.month.isin(list(summer_months)).fillna(False)

        counts_before = s['type_class'].value_counts().to_dict()
        avail = {c: int((s['type_class'] == c).sum()) for c in class_order}
        m_units = min(avail[c] // req[c] for c in class_order)

        if m_units <= 0:
            keep = s.drop(columns=['target_dt', 'is_summer'])
            out.append(keep)
            summary[split_name] = {
                'mode': 'kept_unbalanced_insufficient_rows',
                'rows_before': int(len(s)),
                'rows_after': int(len(keep)),
                'counts_before': counts_before,
                'counts_after': counts_before,
                'required_per_unit': req,
                'available': avail,
            }
            continue

        keep_parts = []
        target = {}
        for c in class_order:
            cls_rows = s[s['type_class'] == c]
            idx = cls_rows.index.to_numpy(dtype=np.int64)
            n_take = int(req[c] * m_units)
            target[c] = n_take

            if split_name == 'train' and float(train_summer_weight) > 1.0:
                w = np.where(cls_rows['is_summer'].to_numpy(), float(train_summer_weight), 1.0)
                choose = _sample_indices_weighted(idx, n_take=n_take, rng=rng, weights=w)
            else:
                choose = _sample_indices_weighted(idx, n_take=n_take, rng=rng, weights=None)

            keep_parts.append(s.loc[choose])

        kept = pd.concat(keep_parts, ignore_index=True)
        kept = kept.sample(frac=1.0, random_state=int(seed) + hash(split_name) % 9973).reset_index(drop=True)
        kept = kept.drop(columns=['target_dt', 'is_summer'])
        out.append(kept)

        summary[split_name] = {
            'mode': 'strict_ratio_downsampled',
            'rows_before': int(len(s)),
            'rows_after': int(len(kept)),
            'm_units': int(m_units),
            'counts_before': counts_before,
            'counts_after': kept['type_class'].value_counts().to_dict(),
            'required_per_unit': req,
            'target_counts': target,
            'available': avail,
            'train_summer_weight': float(train_summer_weight) if split_name == 'train' else 1.0,
        }

    final = pd.concat(out, ignore_index=True) if out else labels.iloc[0:0].copy()
    final = final.sample(frac=1.0, random_state=int(seed)).reset_index(drop=True)
    return final, summary



## Raw Pool Loading and Key Cleaning

The raw fire and no-fire pattern pools are loaded, standardized, deduplicated, and checked for conflicting labels or class assignments at the sample-key level.


In [4]:
# -------------------------
# Load + clean raw pools
# -------------------------
parts = []
for tcls, fn in RAW_FILES.items():
    parts.append(_load_one(LABEL_DIR / fn, tcls))

labels_raw = pd.concat(parts, ignore_index=True)
print('Raw rows:', len(labels_raw))

labels_raw = labels_raw.drop_duplicates(subset=['target_date', 'row', 'col', 'label', 'type_class']).reset_index(drop=True)

g = labels_raw.groupby(KEY_COLS).agg(n_label=('label', 'nunique'), n_class=('type_class', 'nunique')).reset_index()
conflict = g[(g['n_label'] > 1) | (g['n_class'] > 1)][KEY_COLS]
if len(conflict) > 0:
    conflict_set = set(map(tuple, conflict.to_numpy()))
    m = labels_raw[KEY_COLS].apply(lambda r: tuple(r), axis=1).isin(conflict_set)
    labels_clean = labels_raw.loc[~m].copy()
else:
    labels_clean = labels_raw.copy()

labels_clean = labels_clean.sort_values(KEY_COLS).drop_duplicates(subset=KEY_COLS, keep='first').reset_index(drop=True)

print('Clean unique keys:', len(labels_clean))
print('Conflicting keys removed:', len(conflict))
print(labels_clean['type_class'].value_counts())


Raw rows: 58903834
Clean unique keys: 58903834
Conflicting keys removed: 0
type_class
NOFIRE0D     56769409
NOFIRE0A      1587915
NOFIRE0C       265372
NOFIRE0B       177765
FIRETYPE1       59517
FIRETYPE2       25297
FIRETYPE5       18559
Name: count, dtype: int64


## Split Assignment with History Protection

This section assigns train, validation, and test splits according to the year-based summer policy and then removes samples whose 64-day lookback would cross split boundaries.


In [5]:
# -------------------------
# Split policy: val summer years (auto-expand near target), test summer 2023, train others
# Strict 64-day no-mix history guard
# -------------------------
d0 = labels_clean.copy()
d0['target_date'] = pd.to_datetime(d0['target_date'], utc=True)
d0['year'] = d0['target_date'].dt.year.astype(int)
d0['month'] = d0['target_date'].dt.month.astype(int)


def _summer_range(year: int, months: tuple[int, ...]):
    start = pd.Timestamp(f'{year}-{min(months):02d}-01', tz='UTC')
    end = pd.Timestamp(f'{year}-{max(months):02d}-01', tz='UTC') + pd.offsets.MonthEnd(1)
    return start, end


def _build_split_with_guard(base_df: pd.DataFrame, val_years: tuple[int, ...], test_year: int):
    d = base_df.copy()

    val_years = tuple(sorted(int(y) for y in val_years))
    test_year = int(test_year)

    val_ranges = [_summer_range(y, SUMMER_MONTHS) for y in val_years]
    test_range = _summer_range(test_year, SUMMER_MONTHS)
    blocked_ranges = val_ranges + [test_range]

    # Base assignment by target date only
    d['split'] = 'train'
    d['is_summer'] = d['month'].isin(list(SUMMER_MONTHS))
    for y in val_years:
        d.loc[(d['year'] == y) & d['is_summer'], 'split'] = 'val'
    d.loc[(d['year'] == test_year) & d['is_summer'], 'split'] = 'test'

    d['history_start'] = d['target_date'] - pd.Timedelta(days=HISTORY_DAYS)
    d['history_safe'] = False

    # test-safe: target and 64-day lookback both fully inside test summer block
    test_start, test_end = test_range
    mask_test = d['split'] == 'test'
    d.loc[mask_test, 'history_safe'] = (
        (d.loc[mask_test, 'target_date'] >= test_start) &
        (d.loc[mask_test, 'target_date'] <= test_end) &
        (d.loc[mask_test, 'history_start'] >= test_start)
    )

    # val-safe: must fit fully inside one of the val summer blocks
    mask_val = d['split'] == 'val'
    if mask_val.any():
        safe_val = np.zeros(mask_val.sum(), dtype=bool)
        tv = d.loc[mask_val, 'target_date']
        hv = d.loc[mask_val, 'history_start']
        for vs, ve in val_ranges:
            safe_val = safe_val | ((tv >= vs) & (tv <= ve) & (hv >= vs)).to_numpy()
        d.loc[mask_val, 'history_safe'] = safe_val

    # train-safe: no overlap with any blocked summer block (val/test)
    mask_train = d['split'] == 'train'
    if mask_train.any():
        tt = d.loc[mask_train, 'target_date']
        hs = d.loc[mask_train, 'history_start']
        safe_train = np.ones(mask_train.sum(), dtype=bool)
        for bs, be in blocked_ranges:
            overlap = (tt > bs) & (hs <= be)
            safe_train = safe_train & (~overlap.to_numpy())
        d.loc[mask_train, 'history_safe'] = safe_train

    before_guard = len(d)
    d = d[d['history_safe'] == True].copy()
    after_guard = len(d)

    d['target_date'] = d['target_date'].dt.strftime('%Y-%m-%d')
    labels_split_local = d[['target_date', 'row', 'col', 'label', 'type_class', 'split']].reset_index(drop=True)
    labels_split_local = labels_split_local.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    return labels_split_local, before_guard, after_guard


def _estimated_val_rows_after_ratio(labels_split_local: pd.DataFrame) -> int:
    # Estimate final val rows after strict ratio downsample logic.
    if not APPLY_RATIO:
        return int((labels_split_local['split'] == 'val').sum())
    req = _required_per_unit(FIRE_RATIO_232, NOFIRE_RATIO_1121, ENFORCE_FIRE_NOFIRE_1_TO_1)
    class_order = list(req.keys())
    v = labels_split_local[labels_split_local['split'] == 'val']
    if v.empty:
        return 0
    avail = {c: int((v['type_class'] == c).sum()) for c in class_order}
    m_units = min(avail[c] // req[c] for c in class_order)
    if m_units <= 0:
        return int(len(v))
    return int(m_units * sum(req.values()))


# Build candidate val-year sets (most recent summers before test year)
avail_years = sorted(
    int(y) for y in d0.loc[d0['month'].isin(list(SUMMER_MONTHS)), 'year'].unique()
    if int(y) < int(TEST_SUMMER_YEAR)
)
seed_years = tuple(sorted(int(y) for y in VAL_SUMMER_YEARS if int(y) in avail_years))
if not seed_years:
    raise ValueError('Configured VAL_SUMMER_YEARS not found in available summer years before test year')

if AUTO_SELECT_VAL_YEARS:
    desc = sorted(avail_years, reverse=True)
    candidates = [tuple(sorted(desc[:k])) for k in range(1, len(desc) + 1)]

    best = None
    for yrs in candidates:
        ls, b0, a0 = _build_split_with_guard(d0, val_years=yrs, test_year=TEST_SUMMER_YEAR)
        val_est = _estimated_val_rows_after_ratio(ls)
        score = abs(int(val_est) - int(TARGET_VAL_ROWS))
        # tie-breaker: prefer fewer years if equally close
        tie = len(yrs)
        cur = (score, tie, yrs, ls, b0, a0, val_est)
        if (best is None) or ((cur[0], cur[1]) < (best[0], best[1])):
            best = cur

    _, _, VAL_SUMMER_YEARS_USED, labels_split, before_guard, after_guard, val_estimated = best
else:
    VAL_SUMMER_YEARS_USED = seed_years
    labels_split, before_guard, after_guard = _build_split_with_guard(
        d0,
        val_years=VAL_SUMMER_YEARS_USED,
        test_year=TEST_SUMMER_YEAR,
    )
    val_estimated = _estimated_val_rows_after_ratio(labels_split)

print('Split policy: val=summers in', VAL_SUMMER_YEARS_USED, '| test=summer in', TEST_SUMMER_YEAR, '| train=all other dates')
print('Summer months:', SUMMER_MONTHS)
print('Before history guard:', before_guard)
print('After history guard:', after_guard)
print('Estimated val rows after ratio step:', val_estimated, '| target:', TARGET_VAL_ROWS)
print(labels_split.groupby(['split', 'type_class']).size().unstack(fill_value=0))



Split policy: val=summers in (2018, 2019, 2020, 2021, 2022) | test=summer in 2023 | train=all other dates
Summer months: (5, 6, 7, 8)
Before history guard: 58903834
After history guard: 47473580
Estimated val rows after ratio step: 10290 | target: 10000
type_class  FIRETYPE1  FIRETYPE2  FIRETYPE5  NOFIRE0A  NOFIRE0B  NOFIRE0C  \
split                                                                       
test              860        711        409     37953      3102      5324   
train           38866       8666      10199    899462     86790    143306   
val              7515       7691       2214    208225     26773     29047   

type_class  NOFIRE0D  
split                 
test          837032  
train       41027680  
val          4091755  


## Optional Ratio Rebalancing

After split filtering, the class composition can be rebalanced within each split according to the specified fire and no-fire ratios.


In [6]:
# -------------------------
# Optional ratio rebalance per split
# -------------------------
ratio_summary = {}
if APPLY_RATIO:
    labels_final, ratio_summary = _rebalance_per_split(
        labels=labels_split,
        seed=SEED,
        fire_ratio=FIRE_RATIO_232,
        nofire_ratio=NOFIRE_RATIO_1121,
        enforce_1to1=ENFORCE_FIRE_NOFIRE_1_TO_1,
        summer_months=SUMMER_MONTHS,
        train_summer_weight=TRAIN_SUMMER_WEIGHT,
    )
else:
    labels_final = labels_split.copy()

print('Final rows:', len(labels_final))
print(labels_final.groupby(['split', 'type_class']).size().unstack(fill_value=0))



Final rows: 59710
type_class  FIRETYPE1  FIRETYPE2  FIRETYPE5  NOFIRE0A  NOFIRE0B  NOFIRE0C  \
split                                                                       
test              270        270        405       189       189       378   
train            6790       6790      10185      4753      4753      9506   
val              1470       1470       2205      1029      1029      2058   

type_class  NOFIRE0D  
split                 
test             189  
train           4753  
val             1029  


## Export

The finalized combined label table and its run summary are written to disk in this section.


In [7]:
# -------------------------
# Save single combined label file
# -------------------------
# Keep randomized row order (do not sort by timeline)
labels_final = labels_final.sample(frac=1.0, random_state=SEED).reset_index(drop=True)

labels_final.to_csv(OUT_LABELS, index=False)

final_split_class_counts = (
    labels_final
    .groupby(['split', 'type_class'])
    .size()
    .reset_index(name='count')
    .to_dict(orient='records')
)

summary = {
    'seed': int(SEED),
    'history_days_guard': int(HISTORY_DAYS),
    'split_policy': {
        'val_summer_years_seed': [int(y) for y in VAL_SUMMER_YEARS],
        'val_summer_years_used': [int(y) for y in VAL_SUMMER_YEARS_USED],
        'test_summer_year': int(TEST_SUMMER_YEAR),
        'summer_months': list(SUMMER_MONTHS),
        'train_definition': 'all rows not in val/test summer blocks',
        'strict_no_mix_history': True,
        'auto_select_val_years': bool(AUTO_SELECT_VAL_YEARS),
        'target_val_rows': int(TARGET_VAL_ROWS),
        'estimated_val_rows_after_ratio': int(val_estimated),
    },
    'raw_files': list(RAW_FILES),
    'counts': {
        'raw_rows': int(len(labels_raw)),
        'clean_unique_keys': int(len(labels_clean)),
        'before_history_guard': int(before_guard),
        'after_history_guard': int(after_guard),
        'final_rows': int(len(labels_final)),
    },
    'apply_ratio': bool(APPLY_RATIO),
    'ratio_config': {
        'fire_ratio_232': list(FIRE_RATIO_232) if FIRE_RATIO_232 is not None else None,
        'nofire_ratio_1121': list(NOFIRE_RATIO_1121) if NOFIRE_RATIO_1121 is not None else None,
        'enforce_fire_nofire_1_to_1': bool(ENFORCE_FIRE_NOFIRE_1_TO_1),
        'train_summer_weight': float(TRAIN_SUMMER_WEIGHT),
    },
    'ratio_summary': ratio_summary,
    'final_split_class_counts': final_split_class_counts,
    'output_csv': str(OUT_LABELS),
}

OUT_SUMMARY.write_text(json.dumps(summary, indent=2), encoding='utf-8')

print(f"Saved labels: {OUT_LABELS}")
print(f"Saved summary: {OUT_SUMMARY}")



Saved labels: /Users/copperhead07/Computing/DataCollection/RealWork/LABEL/pattern_matches/LABELS_combined.csv
Saved summary: /Users/copperhead07/Computing/DataCollection/RealWork/LABEL/pattern_matches/label_combined_summary.json


## Diagnostic Inspection

The final section performs a compact inspection of the exported label table, including split counts, date ranges, and class distributions.


In [8]:
import pandas as pd
from pathlib import Path

CSV_PATH = Path("/Users/copperhead07/Computing/DataCollection/RealWork/LABEL/pattern_matches/LABELS_combined.csv")
KEY_COLS = ["target_date", "row", "col"]

df = pd.read_csv(CSV_PATH)

# Basic cleanup
df["split"] = df["split"].astype(str).str.lower().str.strip()
df["target_date"] = pd.to_datetime(df["target_date"], errors="coerce")

print("Total rows:", len(df))
print("Columns:", list(df.columns))
print()

# 1) Row-level split counts
row_counts = df["split"].value_counts(dropna=False).rename_axis("split").to_frame("rows")
row_counts["pct_rows"] = (row_counts["rows"] / len(df) * 100).round(2)
print("Row-level split counts:")
print(row_counts)
print()

# 2) Unique sample-level split counts (target_date,row,col)
u = df.drop_duplicates(KEY_COLS + ["split"])
uniq_counts = u["split"].value_counts(dropna=False).rename_axis("split").to_frame("unique_samples")
uniq_counts["pct_unique_samples"] = (uniq_counts["unique_samples"] / len(u) * 100).round(2)
print("Unique-sample split counts:")
print(uniq_counts)
print()

# 3) Date range per split
date_range = (
    df.groupby("split", dropna=False)["target_date"]
      .agg(min_date="min", max_date="max", n_rows="size")
      .sort_index()
)
print("Date range per split:")
print(date_range)
print()

# 4) Class breakdown per split
if "type_class" in df.columns:
    class_counts = (
        df.groupby(["split", "type_class"], dropna=False)
          .size()
          .unstack(fill_value=0)
          .sort_index()
    )
    print("Class counts per split:")
    print(class_counts)
    print()

# 5) Label(0/1) breakdown per split
if "label" in df.columns:
    label_counts = (
        df.groupby(["split", "label"], dropna=False)
          .size()
          .unstack(fill_value=0)
          .sort_index()
    )
    print("Label counts per split:")
    print(label_counts)


Total rows: 59710
Columns: ['target_date', 'row', 'col', 'label', 'type_class', 'split']

Row-level split counts:
        rows  pct_rows
split                 
train  47530     79.60
val    10290     17.23
test    1890      3.17

Unique-sample split counts:
       unique_samples  pct_unique_samples
split                                    
train           47530               79.60
val             10290               17.23
test             1890                3.17

Date range per split:
        min_date   max_date  n_rows
split                              
test  2023-07-04 2023-08-31    1890
train 2012-04-01 2023-12-31   47530
val   2018-07-04 2022-08-31   10290

Class counts per split:
type_class  FIRETYPE1  FIRETYPE2  FIRETYPE5  NOFIRE0A  NOFIRE0B  NOFIRE0C  \
split                                                                       
test              270        270        405       189       189       378   
train            6790       6790      10185      4753      4753      9506